In [ ]:
from __future__ import annotations
import json
import re
from pathlib import Path
import pandas as pd
import argparse
import pathlib
import os

import numpy as np

import matplotlib.pyplot as plt

def set_fontsize(base_fontsize=15):
    fontsize = base_fontsize
    plt.rcParams.update({
        'font.size': fontsize,
        'axes.titlesize': fontsize * 1,
        'axes.labelsize': fontsize,
        'xtick.labelsize': fontsize * 0.8,
        'ytick.labelsize': fontsize * 0.8,
        'legend.fontsize': fontsize * 0.8,
        'font.family': "Arial"
    })

plt.style.use('default')

set_fontsize()

In [ ]:
town = "Mainz"
phase_cut_date = "2023-06-01"
cutoff_value = 0.05 # cutoff value for ensemble member selection (fraction of best models)
objective = "three_objectives"

## Plot predictions for prev for different objectives

In [ ]:
prev_phase_cut_dates = ["2023-01-16", "2023-03-01", "2023-06-01"] # Mainz
#prev_phase_cut_dates = ["2023-01-30", "2023-03-01", "2023-06-01"] # Trier

descriptions = ["1 day", "1.5 months", "3 months"]

In [ ]:
# load pred
prev_data = {}
for prev_phase_cut_date in prev_phase_cut_dates:
    prev_data[prev_phase_cut_date] = np.load(f"/home/iru-mls/marvin/ww_bonn_jax/{town}/multistart_results/{phase_cut_date}_prev{prev_phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_test_positive_rate.npz")


In [ ]:
import jax.numpy as jnp
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

import sys 

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from optimization import optimization_utils

# load data and base model
base_config = {
        # data selection settings
        "data_kwargs": {
            "town": town,
            "log_scale": True, # this only considers WW measurements, not case counts
        },

       "phase_cut_date": phase_cut_date, # date to split data into two phases
        "dt": 0.2,
        "T_max": 25, # dummy value

        "underreporting_model": "monotone_increasing"
}



In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=3, sharex=True, sharey="row", figsize=(13.4, 2.5), dpi=300, constrained_layout=True)

for i, prev_phase_cut_date in enumerate(prev_phase_cut_dates):

    base_config["data_kwargs"]["town"] = town
    base_config["phase_cut_date"] = phase_cut_date
    base_config["prev_phase_cut_date"] = prev_phase_cut_date
    base_config["objective"] = objective
    hparams_path = f"/home/iru-mls/marvin/ww_bonn_jax/{town}/optuna_best_{phase_cut_date}_prev{prev_phase_cut_date}_{objective}/hparams.json"

    with open(hparams_path) as f:
        hparams = json.load(f)

    base_config.update(hparams)
    
    data = optimization_utils.two_phase_integrative_model_load_data(base_config)


    quantiles_prev = {q: jnp.quantile(prev_data[prev_phase_cut_date]["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}
    prev_median = quantiles_prev[0.5]
    print(f"Test MSE for prev phase cut date {descriptions[i]}: {((prev_median[data["t_mask_prev_test"]]-(data["pos_tests_test"]/data["n_tests_test"]*100))**2).mean()}")

    axs[i].scatter(data["prevalence_dates_train"], data["pos_tests_train"]/data["n_tests_train"]*100, color="#63A066", s=15, label="Training/\nValidation")
    axs[i].scatter(data["prevalence_dates_val"], data["pos_tests_val"]/data["n_tests_val"]*100, color="#63A066", s=15, label=None)
    axs[i].scatter(data["prevalence_dates_test"], data["pos_tests_test"]/data["n_tests_test"]*100, color="#595959", s=15, label="Test")
    axs[i].axvline(pd.to_datetime(prev_phase_cut_date), color="#595959", linestyle='--', label="Phase split")

    axs[i].plot(data["dates_all"], prev_median, c="#8B0000", label="Median")

    pos_rate_low  = quantiles_prev[0.25]
    pos_rate_high = quantiles_prev[0.75]
    axs[i].fill_between(data["dates_all"], pos_rate_low, pos_rate_high, color="#8B0000", alpha=0.45, label="50% CI")

    pos_rate_low  = quantiles_prev[0.05]
    pos_rate_high = quantiles_prev[0.95]
    axs[i].fill_between(data["dates_all"], pos_rate_low, pos_rate_high, color="#8B0000", alpha=0.3, label="90% CI")

    pos_rate_low  = quantiles_prev[0.025]
    pos_rate_high = quantiles_prev[0.975]
    axs[i].fill_between(data["dates_all"], pos_rate_low, pos_rate_high, color="#8B0000", alpha=0.15, label="95% CI")


    axs[i].tick_params(axis='x', rotation=45)
    # axs[i].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 5, 9)))
    axs[i].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
    axs[i].set_title(descriptions[i])

axs[0].set_ylabel(f"Test positive\nrate [%]")
plt.tight_layout()
plt.savefig(f"sentisurv_prev_phase_comparison_{town}.png", dpi=300, bbox_inches='tight')

In [ ]:
fig2, ax2 = plt.subplots(figsize=(5, 5), dpi=300)

handles, labels = [], []
for ax in axs.ravel():
    h, l = ax.get_legend_handles_labels()
    for hh, ll in zip(h, l):
        if ll and ll not in labels:
            labels.append(ll)
            handles.append(hh)
#order = [0, 1, 2, 3, 4, 5, 6,]
#labels = [labels[i] for i in order]
#handles = [handles[i] for i in order]

ax2.axis('off')
fig2.legend(handles, labels, frameon=False)

fig2.savefig("sentisurv_objective_comparison/legend.png", dpi=300)